# Stream B — CLIP Features with Real NewsCLIPpings Embeddings

This notebook fixes the previous Stream B issue by using the real NewsCLIPpings embedding PKLs:

- `clip_image_embeddings_{train,val,test}.pkl`
- `clip_text_embeddings_{train,val,test}.pkl`

For VERITE/MMFakeBench, it still computes OpenCLIP embeddings from local processed images.


In [1]:
from pathlib import Path
import sys, os, json, pickle, hashlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES

CLIP_EMB_DIR = STREAM_B_CACHE / "clip_embeddings"
FEATURE_DIR = STREAM_B_CACHE / "features"
NEWS_EMB_ROOT = ROOT / "data" / "raw" / "newsclippings" / "embeddings"

for p in [STREAM_B_CACHE, CLIP_EMB_DIR, FEATURE_DIR, FIGURES, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("NEWS_EMB_ROOT:", NEWS_EMB_ROOT)


ROOT: E:\eve-rag
NEWS_EMB_ROOT: E:\eve-rag\data\raw\newsclippings\embeddings


In [2]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

IMAGE_BATCH_SIZE = 48
TEXT_BATCH_SIZE = 256


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0


In [3]:
def read_manifest(path):
    return pd.read_csv(
        path,
        dtype={
            "pair_id": "string",
            "dataset": "string",
            "split": "string",
            "caption": "string",
            "image_path": "string",
            "image_id": "string",
            "clip_image_key": "string",
            "clip_text_key": "string",
        },
        low_memory=False,
    )

df_all = read_manifest(ALL_MANIFEST)
print("rows:", len(df_all))
print(df_all.groupby(["dataset", "split"])["pair_id"].count())
print("has_image:", int(df_all["has_image"].sum()))


rows: 97361
dataset        split
mmfakebench    test     10000
               val       1000
newsclippings  test      7264
               train    71072
               val       7024
verite         test      1001
Name: pair_id, dtype: int64
has_image: 11621


In [4]:
def load_newsclip_embedding_dict(kind, split):
    p = NEWS_EMB_ROOT / kind / f"{kind}_{split}.pkl"
    if not p.exists():
        raise FileNotFoundError(p)
    obj = pickle.load(open(p, "rb"))
    if not isinstance(obj, dict):
        raise TypeError(f"Unsupported embedding format in {p}: {type(obj)}")
    return {str(k): np.asarray(v, dtype=np.float32) for k, v in obj.items()}

news_img_embs = {}
news_txt_embs = {}

for split in ["train", "val", "test"]:
    news_img_embs[split] = load_newsclip_embedding_dict("clip_image_embeddings", split)
    news_txt_embs[split] = load_newsclip_embedding_dict("clip_text_embeddings", split)

for split in ["train", "val", "test"]:
    print(split, "image:", len(news_img_embs[split]), "text:", len(news_txt_embs[split]))


train image: 473663 text: 385850
val image: 50000 text: 40864
test image: 50000 text: 40915


In [5]:
try:
    import open_clip
except Exception as e:
    raise RuntimeError("Install open_clip_torch: pip install open_clip_torch") from e

MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE).eval()

print("Loaded OpenCLIP:", MODEL_NAME, PRETRAINED)


Loaded OpenCLIP: ViT-B-32 openai


In [6]:
from PIL import Image
from tqdm.auto import tqdm

@torch.no_grad()
def encode_texts(texts, batch_size=TEXT_BATCH_SIZE):
    if not texts:
        return np.zeros((0, 512), dtype=np.float32)
    outs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="OpenCLIP text"):
        toks = tokenizer(texts[i:i+batch_size]).to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            z = model.encode_text(toks).float()
        z = z / z.norm(dim=-1, keepdim=True)
        outs.append(z.cpu().numpy().astype(np.float32))
        del toks, z
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)

@torch.no_grad()
def encode_images(paths, batch_size=IMAGE_BATCH_SIZE):
    if not paths:
        return np.zeros((0, 512), dtype=np.float32)
    outs = []
    for i in tqdm(range(0, len(paths), batch_size), desc="OpenCLIP image"):
        ims = []
        for p in paths[i:i+batch_size]:
            try:
                ims.append(preprocess(Image.open(p).convert("RGB")))
            except Exception:
                ims.append(torch.zeros(3, 224, 224))
        ims = torch.stack(ims).to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            z = model.encode_image(ims).float()
        z = z / z.norm(dim=-1, keepdim=True)
        outs.append(z.cpu().numpy().astype(np.float32))
        del ims, z
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)

def norm_vec(v):
    v = np.asarray(v, dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

def stable_hash_vec(key, dim=512):
    h = hashlib.sha256(str(key).encode("utf-8")).digest()
    rng = np.random.default_rng(int.from_bytes(h[:8], "little"))
    v = rng.normal(size=dim).astype(np.float32)
    return norm_vec(v)


In [7]:
# OpenCLIP text embeddings for VERITE/MMFakeBench and fallback.
REBUILD_TEXT_EMBS = False

text_emb_path = CLIP_EMB_DIR / "text_embs_all.npy"
text_ids_path = CLIP_EMB_DIR / "text_pair_ids_all.json"

if text_emb_path.exists() and text_ids_path.exists() and not REBUILD_TEXT_EMBS:
    text_embs = np.load(text_emb_path)
    text_pair_ids = json.load(open(text_ids_path, "r", encoding="utf-8"))
    print("Loaded OpenCLIP text embeddings:", text_embs.shape)
else:
    text_pair_ids = df_all["pair_id"].astype(str).tolist()
    captions = df_all["caption"].fillna("").astype(str).tolist()
    text_embs = encode_texts(captions)
    np.save(text_emb_path, text_embs)
    json.dump(text_pair_ids, open(text_ids_path, "w", encoding="utf-8"))
    print("Saved OpenCLIP text embeddings:", text_embs.shape)

text_idx = {pid: i for i, pid in enumerate(text_pair_ids)}


OpenCLIP text:   0%|          | 0/381 [00:00<?, ?it/s]

Saved OpenCLIP text embeddings: (97361, 512)


In [8]:
# OpenCLIP local image embeddings for VERITE/MMFakeBench.
REBUILD_IMAGE_EMBS = False

local_imgs = df_all[df_all["has_image"] == 1][["image_id", "image_path"]].drop_duplicates("image_id").copy()
local_imgs["abs_image_path"] = local_imgs["image_path"].map(lambda p: ROOT / str(p))
local_imgs = local_imgs[local_imgs["abs_image_path"].map(lambda p: p.exists())].reset_index(drop=True)

image_emb_path = CLIP_EMB_DIR / "image_embs_local.npy"
image_ids_path = CLIP_EMB_DIR / "image_ids_local.json"

if image_emb_path.exists() and image_ids_path.exists() and not REBUILD_IMAGE_EMBS:
    image_embs_local = np.load(image_emb_path)
    image_ids_local = json.load(open(image_ids_path, "r", encoding="utf-8"))
    print("Loaded local image embeddings:", image_embs_local.shape)
else:
    image_ids_local = local_imgs["image_id"].astype(str).tolist()
    image_embs_local = encode_images(local_imgs["abs_image_path"].tolist())
    np.save(image_emb_path, image_embs_local)
    json.dump(image_ids_local, open(image_ids_path, "w", encoding="utf-8"))
    print("Saved local image embeddings:", image_embs_local.shape)

image_idx = {iid: i for i, iid in enumerate(image_ids_local)}


OpenCLIP image:   0%|          | 0/240 [00:00<?, ?it/s]

Saved local image embeddings: (11495, 512)


In [9]:
def get_newsclip_image_emb(row):
    return news_img_embs.get(str(row.split), {}).get(str(row.clip_image_key))

def get_newsclip_text_emb(row):
    return news_txt_embs.get(str(row.split), {}).get(str(row.clip_text_key))

rows = []

missing_news_img = 0
missing_news_txt = 0
used_real_newsclip = 0
used_real_local = 0
used_fallback = 0

for r in tqdm(df_all.itertuples(index=False), total=len(df_all), desc="Building features"):
    pid = str(r.pair_id)
    dataset = str(r.dataset)
    iid = str(r.image_id)

    if dataset == "newsclippings":
        ivec = get_newsclip_image_emb(r)
        tvec = get_newsclip_text_emb(r)

        if ivec is None:
            missing_news_img += 1
        if tvec is None:
            missing_news_txt += 1

        if ivec is not None and tvec is not None:
            has_real_image_emb = 1
            used_real_newsclip += 1
        else:
            if tvec is None:
                tvec = text_embs[text_idx[pid]]
            if ivec is None:
                ivec = stable_hash_vec(str(r.clip_image_key) if str(r.clip_image_key) else iid)
            has_real_image_emb = 0
            used_fallback += 1

    else:
        tvec = text_embs[text_idx[pid]]
        if int(r.has_image) == 1 and iid in image_idx:
            ivec = image_embs_local[image_idx[iid]]
            has_real_image_emb = 1
            used_real_local += 1
        else:
            ivec = stable_hash_vec(iid)
            has_real_image_emb = 0
            used_fallback += 1

    ivec = norm_vec(ivec)
    tvec = norm_vec(tvec)

    cos = float(np.dot(ivec, tvec))
    diff = np.abs(ivec - tvec)
    prod = ivec * tvec

    rows.append({
        "pair_id": pid,
        "dataset": r.dataset,
        "split": r.split,
        "label": int(r.label),
        "has_image": int(r.has_image),
        "has_real_image_emb": has_real_image_emb,
        "clip_cosine": cos,
        "l2_dist": float(np.linalg.norm(ivec - tvec)),
        "absdiff_mean": float(diff.mean()),
        "absdiff_max": float(diff.max()),
        "prod_mean": float(prod.mean()),
        "text_norm": float(np.linalg.norm(tvec)),
        "image_norm": float(np.linalg.norm(ivec)),
    })

features = pd.DataFrame(rows)

features.to_csv(FEATURE_DIR / "clip_features_all.csv", index=False)
pickle.dump(features, open(FEATURE_DIR / "clip_features_all.pkl", "wb"), protocol=4)

for (dataset, split), sub in features.groupby(["dataset", "split"]):
    sub.to_csv(FEATURE_DIR / f"clip_features_{dataset}_{split}.csv", index=False)

print("Feature rows:", features.shape)
print("Used real NewsCLIP embeddings:", used_real_newsclip)
print("Used real local image embeddings:", used_real_local)
print("Used fallback:", used_fallback)
print("Missing NewsCLIP image embeddings:", missing_news_img)
print("Missing NewsCLIP text embeddings:", missing_news_txt)

summary = features.groupby(["dataset", "split"]).agg(
    rows=("pair_id", "count"),
    has_image=("has_image", "sum"),
    has_real_image_emb=("has_real_image_emb", "sum"),
    clip_cosine_mean=("clip_cosine", "mean"),
    clip_cosine_std=("clip_cosine", "std"),
).reset_index()

summary.to_csv(TABLES / "stream_b_clip_feature_summary.csv", index=False)
summary


Building features:   0%|          | 0/97361 [00:00<?, ?it/s]

Feature rows: (97361, 13)
Used real NewsCLIP embeddings: 85360
Used real local image embeddings: 11621
Used fallback: 380
Missing NewsCLIP image embeddings: 0
Missing NewsCLIP text embeddings: 0


,dataset,split,rows,has_image,has_real_image_emb,clip_cosine_mean,clip_cosine_std
0,mmfakebench,test,10000,10000,10000,0.283725,0.040580
1,mmfakebench,val,1000,1000,1000,0.287078,0.039972
2,newsclippings,test,7264,0,7264,0.290589,0.043445
3,newsclippings,train,71072,0,71072,0.290366,0.043841
4,newsclippings,val,7024,0,7024,0.291129,0.043007
5,verite,test,1001,621,621,0.113193,0.084352


In [10]:
print("=" * 60)
print("CLIP features complete.")
print("Good expected signs:")
print("  Used real NewsCLIP embeddings: 85360")
print("  Missing NewsCLIP image embeddings: 0")
print("  Missing NewsCLIP text embeddings: 0")
print("  Used fallback: around 380, mostly VERITE rows without local images")
print("Next: rerun lora_stage1_updated.ipynb")
print("=" * 60)


CLIP features complete.
Good expected signs:
  Used real NewsCLIP embeddings: 85360
  Missing NewsCLIP image embeddings: 0
  Missing NewsCLIP text embeddings: 0
  Used fallback: around 380, mostly VERITE rows without local images
Next: rerun lora_stage1_updated.ipynb
